# ByT5 IPA G2P Training (Kaggle)

Fine-tunes `google/byt5-small` on (headword → IPA) pairs for the LexCW app.

### Setup
1. **Add `pairs.json` as a dataset** — sidebar → + Add Data → Upload
2. **Enable GPU** — Settings → Accelerator → **GPU P100** (preferred: best fp32 throughput and avoids the dual-T4 DataParallel slowdown; T4 x2 also works since we pin to one GPU)
3. **Run cells in order** — ~5–15 min with early stopping (was hours: the old config ran autoregressive generation every epoch and wrote ~3.6 GB of checkpoints per epoch)
4. Download the zip at the end, unzip into `instance/ipa_models/` on the server

Training runs fp32: byT5/T5 produces NaN losses in fp16, and Kaggle GPUs (T4/P100) have no bf16. Model selection uses `eval_loss`; sample generations are printed once after training.

In [ ]:
import os, subprocess

# Use a single GPU. Kaggle's "T4 x2" makes HF Trainer fall into nn.DataParallel,
# which is slower than one GPU for a model this small (replication + gather every step).
# Must be set before torch is imported anywhere.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# Find pairs.json wherever Kaggle placed it
result = subprocess.run(['find', '/kaggle/input', '-name', 'pairs.json'], capture_output=True, text=True)
paths = result.stdout.strip().split('\n')
if paths and paths[0]:
    PAIRS_JSON = paths[0]
else:
    PAIRS_JSON = 'pairs.json'
print(f'Using pairs file: {PAIRS_JSON}')
os.environ['PAIRS_JSON'] = PAIRS_JSON

In [ ]:
!pip install -q -U transformers datasets accelerate
print('Dependencies OK')

In [ ]:
import json, os, time, warnings
warnings.filterwarnings('ignore')

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)

# ---- Config ----
MODEL_NAME = 'google/byt5-small'
IPA_WS = 'seh-fonipa'
# Byte-level: headwords are <=24 bytes, IPA <=62 bytes in our data. 128 wasted
# padding compute; 80 leaves headroom for longer future entries.
MAX_LENGTH = 80
OUTPUT_DIR = './byt5_model'

EPOCHS = 100          # early stopping usually ends it well before this
BATCH_SIZE = 16
LEARNING_RATE = 5e-4

# ---- Load data ----
pairs_file = os.environ.get('PAIRS_JSON', 'pairs.json')
with open(pairs_file, encoding='utf-8') as f:
    raw = json.load(f)

pairs = [(d['headword'], d['ipa']) for d in raw if d.get('headword') and d.get('ipa')]
print(f'Loaded {len(pairs)} pairs')

if len(pairs) < 10:
    print('ERROR: too few pairs')
    raise SystemExit(2)

sources = [h for h, _ in pairs]
targets = [i for _, i in pairs]

# ---- Tokenize ----
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')

def preprocess(batch):
    model_inputs = tok(batch['source'], max_length=MAX_LENGTH, truncation=True)
    labels = tok(text_target=batch['target'], max_length=MAX_LENGTH, truncation=True)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

dataset = Dataset.from_dict({'source': sources, 'target': targets})
dataset = dataset.train_test_split(test_size=0.1, seed=42)
tokenized = dataset.map(preprocess, batched=True, remove_columns=['source', 'target'])
print(f'Train: {len(tokenized["train"])}  Eval: {len(tokenized["test"])}')

data_collator = DataCollatorForSeq2Seq(tok, model=model, padding=True)

# ---- Training args ----
# fp16=False: byT5/T5 is known to produce NaN gradients in fp16; the T4 has no
# bf16, so we stay in fp32 (the model is small enough that this is fine).
#
# Speed-critical choices (this notebook used to take hours on Kaggle):
# - predict_with_generate=False: eval on loss only. Per-epoch autoregressive
#   generation was the main slowdown; eval_loss tracks quality just as well
#   for model selection, and we generate once at the end for inspection.
# - save_only_model=True: skips optimizer/scheduler state, cutting checkpoint
#   writes from ~3.6 GB to ~1.2 GB per epoch on Kaggle's slow disk.
# - EarlyStoppingCallback: stops when eval_loss plateaus instead of always
#   running all 100 epochs.
args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    save_only_model=True,
    predict_with_generate=False,
    fp16=False,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['test'],
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=10)],
)

t0 = time.time()
trainer.train()
print(f'\nTraining took {(time.time() - t0) / 60:.1f} min')

# ---- Test sample ----
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

test_words = ['acceptance test', 'book test', 'attest', 'attestation', 'citizen test']
for w in test_words:
    ids = tok(w, return_tensors='pt').to(device)
    with torch.no_grad():
        gen = model.generate(**ids, num_beams=4, max_length=MAX_LENGTH, early_stopping=True)
    print(f'  {w:30s} -> {tok.decode(gen[0], skip_special_tokens=True)}')

# ---- Save ----
save_path = f'{OUTPUT_DIR}/ipa_byt5_{IPA_WS}'
model.save_pretrained(save_path)
tok.save_pretrained(save_path)

with open(f'{save_path}/metadata.json', 'w') as f:
    json.dump({'ipa_writing_system': IPA_WS, 'base_model': MODEL_NAME,
               'source_prefix': '', 'task': 'g2p'}, f, indent=2)
print(f'\nSaved to: {save_path}')

# ---- Zip ----
import subprocess
archive_name = f'ipa_byt5_{IPA_WS}'
subprocess.run(['zip', '-r', f'{archive_name}.zip', archive_name], cwd=OUTPUT_DIR, check=True)
archive = f'{OUTPUT_DIR}/{archive_name}.zip'
size_mb = os.path.getsize(archive) / 1024 / 1024
print(f'\nZip ready: {archive} ({size_mb:.1f} MB)')

In [ ]:
from IPython.display import FileLink
import os

archive = 'byt5_model/ipa_byt5_seh-fonipa.zip'
if os.path.exists(archive):
    display(FileLink(archive))
    print('Click the link above to download. Then unzip into instance/ipa_models/ on the server.')
else:
    print('Zip not found — check errors above')